#### Loading the Libraries

In [11]:
import pandas as pd
import numpy as np

from joblib import load

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

#### Loading the Test Dataset

In [12]:
test_data = pd.read_csv(
    '../data/processed/test_data.csv'
)

test_data.shape

(2441, 23)

In [13]:
X_test = test_data.drop(
    'Revenue',
    axis=1
)

y_test = test_data['Revenue']

#### Loading the Saved Models

In [14]:
logistic_pipeline = load(
    '../models/logistic_regression_pipeline.joblib'
)

ridge_pipeline = load(
    '../models/ridge_classifier_pipeline.joblib'
)

decision_tree_pipeline = load(
    '../models/decision_tree_classifier_pipeline.joblib'
)

random_forest_pipeline = load(
    '../models/random_forest_classifier_pipeline.joblib'
)

tuned_random_forest_pipeline = load(
    '../models/tuned_random_forest_classifier_pipeline.joblib'
)

extra_trees_pipeline = load(
    '../models/extra_trees_classifier_pipeline.joblib'
)

tuned_extra_trees_pipeline = load(
    '../models/tuned_extra_trees_classifier_pipeline.joblib'
)

gradient_boosting_pipeline = load(
    '../models/gradient_boosting_classifier_pipeline.joblib'
)

tuned_gradient_boosting_pipeline = load(
    '../models/tuned_gradient_boosting_classifier_pipeline.joblib'
)

hist_gradient_boosting_pipeline = load(
    '../models/hist_gradient_boosting_classifier_pipeline.joblib'
)

tuned_hist_gradient_boosting_pipeline = load(
    '../models/tuned_hist_gradient_boosting_classifier_pipeline.joblib'
)

xgboost_pipeline = load(
    '../models/xgboost_classifier_pipeline.joblib'
)

tuned_xgboost_pipeline = load(
    '../models/tuned_xgboost_classifier_pipeline.joblib'
)

voting_pipeline = load(
    '../models/voting_classifier_pipeline.joblib'
)

stacking_pipeline = load(
    '../models/stacking_classifier_pipeline.joblib'
)

#### Generating Test Predictions

In [15]:
models = {
    'Logistic Regression': logistic_pipeline,
    'Ridge Classifier': ridge_pipeline,
    'Decision Tree': decision_tree_pipeline,
    'Random Forest': random_forest_pipeline,
    'Tuned Random Forest': tuned_random_forest_pipeline,
    'Extra Trees': extra_trees_pipeline,
    'Tuned Extra Trees': tuned_extra_trees_pipeline,
    'Gradient Boosting': gradient_boosting_pipeline,
    'Tuned Gradient Boosting': tuned_gradient_boosting_pipeline,
    'HistGradientBoosting': hist_gradient_boosting_pipeline,
    'Tuned HistGradientBoosting': tuned_hist_gradient_boosting_pipeline,
    'XGBoost': xgboost_pipeline,
    'Tuned XGBoost': tuned_xgboost_pipeline,
    'Voting Classifier': voting_pipeline,
    'Stacking Classifier': stacking_pipeline
}

In [16]:
for name, model in models.items():
    try:
        model.predict(X_test)
        print(f"{name}: fitted")
    except Exception as e:
        print(f"{name}: NOT FITTED")

Logistic Regression: fitted
Ridge Classifier: fitted
Decision Tree: fitted
Random Forest: fitted
Tuned Random Forest: fitted
Extra Trees: fitted
Tuned Extra Trees: fitted
Gradient Boosting: fitted
Tuned Gradient Boosting: fitted
HistGradientBoosting: fitted
Tuned HistGradientBoosting: fitted
XGBoost: fitted
Tuned XGBoost: fitted
Voting Classifier: fitted
Stacking Classifier: fitted


In [17]:
predictions = {}

for name, model in models.items():
    predictions[name] = model.predict(X_test)

print("Test predictions generated for all models.")

Test predictions generated for all models.


In [18]:
for name, prediction in predictions.items():
    print(
        f"{name}: {len(prediction)} predictions"
    )

Logistic Regression: 2441 predictions
Ridge Classifier: 2441 predictions
Decision Tree: 2441 predictions
Random Forest: 2441 predictions
Tuned Random Forest: 2441 predictions
Extra Trees: 2441 predictions
Tuned Extra Trees: 2441 predictions
Gradient Boosting: 2441 predictions
Tuned Gradient Boosting: 2441 predictions
HistGradientBoosting: 2441 predictions
Tuned HistGradientBoosting: 2441 predictions
XGBoost: 2441 predictions
Tuned XGBoost: 2441 predictions
Voting Classifier: 2441 predictions
Stacking Classifier: 2441 predictions


#### Calculating Evaluation Metrics

In [19]:
evaluation_results = []

for name, model in models.items():

    y_pred = predictions[name]

    evaluation_results.append({
        'Model': name,
        'Accuracy': accuracy_score(
            y_test,
            y_pred
        ),
        'Precision': precision_score(
            y_test,
            y_pred,
            zero_division=0
        ),
        'Recall': recall_score(
            y_test,
            y_pred,
            zero_division=0
        ),
        'F1-Score': f1_score(
            y_test,
            y_pred,
            zero_division=0
        )
    })

evaluation_results = pd.DataFrame(
    evaluation_results
)

evaluation_results.sort_values(
    'F1-Score',
    ascending=False
).reset_index(drop=True)

,Model,Accuracy,Precision,Recall,F1-Score
0,Gradient Boosting,0.898402,0.655814,0.738220,0.694581
1,Tuned Gradient Boosting,0.893486,0.632609,0.761780,0.691211
2,Tuned XGBoost,0.894306,0.639013,0.746073,0.688406
3,Tuned Random Forest,0.897583,0.659420,0.714660,0.685930
4,Tuned HistGradientBoosting,0.891438,0.630290,0.740838,0.681107
5,Voting Classifier,0.890619,0.627494,0.740838,0.679472
6,Random Forest,0.900860,0.688172,0.670157,0.679045
7,Stacking Classifier,0.896354,0.661654,0.691099,0.676056
8,HistGradientBoosting,0.895944,0.664103,0.678010,0.670984
9,XGBoost,0.894306,0.666667,0.649215,0.657825


#### Important

We're intentionally using F1-score as the primary ranking metric because the Revenue=True class is substantially smaller than Revenue=False.

But we're keeping Accuracy, Precision, and Recall visible so we can understand the trade-offs rather than blindly selecting the highest F1.

#### observations:

- Gradient Boosting beats its tuned version on F1: 0.6946 vs 0.6912.
- Tuned Gradient Boosting has the highest recall at 0.7618, meaning it catches the most purchasing sessions, but at the cost of precision.
- Random Forest has the highest accuracy at 0.9009, but its F1 is lower than Gradient Boosting.
- The ensemble models didn't outperform the best individual boosting models.
- Extra Trees is clearly weaker on this test set, especially recall.

So far, Gradient Boosting is our leading candidate, but we're not selecting it yet.

#### Calculating ROC-AUC Scores

In [20]:
roc_auc_results = []

for name, model in models.items():

    if hasattr(model, 'predict_proba'):
        y_score = model.predict_proba(X_test)[:, 1]

    else:
        y_score = model.decision_function(X_test)

    roc_auc_results.append({
        'Model': name,
        'ROC-AUC': roc_auc_score(
            y_test,
            y_score
        )
    })

roc_auc_results = pd.DataFrame(
    roc_auc_results
)

roc_auc_results.sort_values(
    'ROC-AUC',
    ascending=False
).reset_index(drop=True)

,Model,ROC-AUC
0,Tuned HistGradientBoosting,0.936369
1,Tuned XGBoost,0.935302
2,Tuned Gradient Boosting,0.932349
3,Gradient Boosting,0.931974
4,Voting Classifier,0.931487
5,HistGradientBoosting,0.931304
6,Tuned Random Forest,0.926278
7,XGBoost,0.925086
8,Random Forest,0.920738
9,Stacking Classifier,0.920083


#### Comparing Evaluation Metrics

In [21]:
evaluation_summary = (
    evaluation_results
    .merge(
        roc_auc_results,
        on='Model'
    )
    .sort_values(
        'F1-Score',
        ascending=False
    )
    .reset_index(drop=True)
)

evaluation_summary

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Gradient Boosting,0.898402,0.655814,0.738220,0.694581,0.931974
1,Tuned Gradient Boosting,0.893486,0.632609,0.761780,0.691211,0.932349
2,Tuned XGBoost,0.894306,0.639013,0.746073,0.688406,0.935302
3,Tuned Random Forest,0.897583,0.659420,0.714660,0.685930,0.926278
4,Tuned HistGradientBoosting,0.891438,0.630290,0.740838,0.681107,0.936369
5,Voting Classifier,0.890619,0.627494,0.740838,0.679472,0.931487
6,Random Forest,0.900860,0.688172,0.670157,0.679045,0.920738
7,Stacking Classifier,0.896354,0.661654,0.691099,0.676056,0.920083
8,HistGradientBoosting,0.895944,0.664103,0.678010,0.670984,0.931304
9,XGBoost,0.894306,0.666667,0.649215,0.657825,0.925086


In [22]:
evaluation_summary_plot = (
    evaluation_summary
    .sort_values(
        'F1-Score',
        ascending=True
    )
)

evaluation_summary_plot[
    ['Model', 'F1-Score', 'ROC-AUC']
]

,Model,F1-Score,ROC-AUC
14,Extra Trees,0.525316,0.895214
13,Ridge Classifier,0.527256,0.858651
12,Decision Tree,0.565432,0.751414
11,Tuned Extra Trees,0.567455,0.892921
10,Logistic Regression,0.617108,0.908208
9,XGBoost,0.657825,0.925086
8,HistGradientBoosting,0.670984,0.931304
7,Stacking Classifier,0.676056,0.920083
6,Random Forest,0.679045,0.920738
5,Voting Classifier,0.679472,0.931487


#### Generating Confusion Matrices

In [23]:
top_models = (
    evaluation_summary
    .head(5)['Model']
    .tolist()
)

top_models

['Gradient Boosting',
 'Tuned Gradient Boosting',
 'Tuned XGBoost',
 'Tuned Random Forest',
 'Tuned HistGradientBoosting']

In [24]:
confusion_matrices = {}

for name in top_models:

    confusion_matrices[name] = confusion_matrix(
        y_test,
        predictions[name]
    )

confusion_matrices

{'Gradient Boosting': array([[1911,  148],
        [ 100,  282]]),
 'Tuned Gradient Boosting': array([[1890,  169],
        [  91,  291]]),
 'Tuned XGBoost': array([[1898,  161],
        [  97,  285]]),
 'Tuned Random Forest': array([[1918,  141],
        [ 109,  273]]),
 'Tuned HistGradientBoosting': array([[1893,  166],
        [  99,  283]])}

The important business trade-off is:

- Gradient Boosting has the best F1, with a balanced 148 FP / 100 FN profile.
- Tuned Gradient Boosting catches the most actual purchasers (291 TP) and has the fewest missed purchasers (91 FN), but generates more false positives (169 FP).
- Tuned Random Forest is the most conservative of these five, with only 141 FP, but misses more purchasers (109 FN).

So there isn't a single universally "best" model — it depends on whether missing a potential purchaser or unnecessarily targeting a non-purchaser is more costly.

But for this project, F1 currently gives Gradient Boosting the strongest overall balance.

#### Reviewing Classification Reports

In [25]:
for name in top_models:

    print(
        f"\n{name}"
    )

    print(
        classification_report(
            y_test,
            predictions[name],
            target_names=[
                'No Purchase',
                'Purchase'
            ],
            zero_division=0
        )
    )


Gradient Boosting
              precision    recall  f1-score   support

 No Purchase       0.95      0.93      0.94      2059
    Purchase       0.66      0.74      0.69       382

    accuracy                           0.90      2441
   macro avg       0.80      0.83      0.82      2441
weighted avg       0.90      0.90      0.90      2441


Tuned Gradient Boosting
              precision    recall  f1-score   support

 No Purchase       0.95      0.92      0.94      2059
    Purchase       0.63      0.76      0.69       382

    accuracy                           0.89      2441
   macro avg       0.79      0.84      0.81      2441
weighted avg       0.90      0.89      0.90      2441


Tuned XGBoost
              precision    recall  f1-score   support

 No Purchase       0.95      0.92      0.94      2059
    Purchase       0.64      0.75      0.69       382

    accuracy                           0.89      2441
   macro avg       0.80      0.83      0.81      2441
weighted avg   

The differences are small, but they tell an important story:

- Gradient Boosting provides the strongest overall balance.
- Tuned Gradient Boosting sacrifices precision for higher recall.
- Tuned XGBoost behaves similarly, with slightly lower precision than the original Gradient Boosting.
- Tuned Random Forest has the same precision as Gradient Boosting but lower recall.
- Tuned HistGradientBoosting falls slightly behind on F1.

And importantly, Gradient Boosting's ROC-AUC of 0.9320 is still excellent, even though the tuned HistGradientBoosting and tuned XGBoost rank higher on ROC-AUC.

So our conclusion is not simply:

"Gradient Boosting has the highest score."

It's:

Gradient Boosting provides the best overall balance between identifying purchasing sessions and limiting incorrect purchase predictions at the selected classification threshold.

#### Comparing ROC Curves